In [2]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os

log = load_log()
print(f"Log loaded. Rows: {len(log)}")

Log loaded. Rows: 2


## WJP Pipeline

**Source:** World Justice Project Rule of Law Index
**Access:** Automated direct download — no manual step required
**Download instructions:** See `docs/instructions_data_maintenance.md` — WJP section

### Framework usage
| Factor | Concept | Role |
|--------|---------|------|
| Factor 2: Absence of Corruption | Control of corruption | Primary tier 1 |
| Factor 3: Open Government | Legal quality + Government transparency | Primary tier 1 |
| Factor 4: Fundamental Rights | Legal quality | Primary tier 1 |
| Factor 5: Order and Security | Personal security + Political stability | Primary tier 1 / tier 2 |
| Factor 6: Regulatory Enforcement | Regulatory quality | Primary tier 1 |
| Factor 7: Civil Justice | Judicial independence | Primary tier 1 |
| Factor 8: Criminal Justice | Judicial independence | Primary tier 1 |

In [7]:
import requests
import io
from datetime import datetime

# WJP indicators used in framework by concept
WJP_INDICATORS = {
    'Factor 2: Absence of Corruption':   'wjp_f2_absence_corruption',
    'Factor 3: Open Government':          'wjp_f3_open_government',
    'Factor 4: Fundamental Rights':       'wjp_f4_fundamental_rights',
    'Factor 5: Order and Security':       'wjp_f5_order_security',
    'Factor 6: Regulatory Enforcement':   'wjp_f6_regulatory_enforcement',
    'Factor 7: Civil Justice':            'wjp_f7_civil_justice',
    'Factor 8: Criminal Justice':         'wjp_f8_criminal_justice',
}

def get_latest_wjp_year():
    """Try current year, fall back to prior years until a valid Excel file is found."""
    current_year = datetime.today().year
    for year in range(current_year, current_year - 3, -1):
        url = f"https://worldjusticeproject.org/rule-of-law-index/downloads/{year}_wjp_rule_of_law_index_HISTORICAL_DATA_FILE.xlsx"
        response = requests.get(url)
        # Validate it's actually an Excel file, not an HTML error page
        if response.status_code == 200 and 'text/html' not in response.headers.get('Content-Type', ''):
            return year, url, response.content
    return None, None, None

WJP_YEAR, WJP_URL, WJP_CONTENT = get_latest_wjp_year()
print(f"Latest WJP year: {WJP_YEAR}")
print(f"URL: {WJP_URL}")
print(f"Size: {len(WJP_CONTENT)/1024:.1f} KB")

Latest WJP year: 2025
URL: https://worldjusticeproject.org/rule-of-law-index/downloads/2025_wjp_rule_of_law_index_HISTORICAL_DATA_FILE.xlsx
Size: 2124.8 KB


In [8]:
import io

# Load Excel file from memory
xl = pd.ExcelFile(io.BytesIO(WJP_CONTENT), engine='openpyxl')
print(f"Sheets: {xl.sheet_names}")

# Preview first sheet
df_preview = xl.parse(xl.sheet_names[0])
print(f"\nFirst sheet shape: {df_preview.shape}")
print(df_preview.head())

Sheets: ['Introduction', 'WJP ROL Index 2012-2013 Scores', 'WJP ROL Index 2014 Scores', 'WJP ROL Index 2015 Scores', 'WJP ROL Index 2016 Scores', 'WJP ROL Index 2017-2018 Scores', 'WJP ROL Index 2019 Scores', 'WJP ROL Index 2020 Scores', 'WJP ROL Index 2021 Scores', 'WJP ROL Index 2022 Scores', 'WJP ROL Index 2023 Scores', 'WJP ROL Index 2024 Scores', 'WJP ROL Index 2025 Scores', 'Historical Data']

First sheet shape: (2, 8)
   Unnamed: 0  Unnamed: 1  Unnamed: 2  Unnamed: 3  Unnamed: 4  Unnamed: 5  \
0         NaN         NaN         NaN         NaN         NaN         NaN   
1         NaN         NaN         NaN         NaN         NaN         NaN   

   Unnamed: 6    The World Justice Project  
0         NaN       Rule of Law Index 2025  
1         NaN  www.worldjusticeproject.org  


In [10]:
# Columns to keep — identifiers plus framework factors
# Note: keeping sub-indicators for F6 (6.5 needed for Property Rights) and F7/F8
KEEP_COLS = [
    'Country', 'Country Code', 'Year',
    'Factor 2: Absence of Corruption',
    'Factor 3: Open Government ',           # note trailing space in source
    'Factor 4: Fundamental Rights',
    'Factor 5: Order and Security',
    'Factor 6: Regulatory Enforcement',
    '6.5 The government does not expropriate without lawful process and adequate compensation',
    'Factor 7: Civil Justice',
    'Factor 8: Criminal Justice',
]

# Filter columns
wjp = df_hist[KEEP_COLS].copy()

# Strip trailing spaces from column names
wjp.columns = wjp.columns.str.strip()

# Rename columns
wjp = wjp.rename(columns={
    'Country':                      'country_name',
    'Country Code':                 'country_code',
    'Year':                         'year_str',
    'Factor 2: Absence of Corruption':   'wjp_f2_absence_corruption',
    'Factor 3: Open Government':         'wjp_f3_open_government',
    'Factor 4: Fundamental Rights':      'wjp_f4_fundamental_rights',
    'Factor 5: Order and Security':      'wjp_f5_order_security',
    'Factor 6: Regulatory Enforcement':  'wjp_f6_regulatory_enforcement',
    '6.5 The government does not expropriate without lawful process and adequate compensation': 'wjp_f6_5_no_expropriation',
    'Factor 7: Civil Justice':           'wjp_f7_civil_justice',
    'Factor 8: Criminal Justice':        'wjp_f8_criminal_justice',
})

# Clean year column — some entries are '2012-2013', standardise to end year
wjp['year'] = wjp['year_str'].apply(
    lambda x: int(str(x).split('-')[-1]) if '-' in str(x) else int(x)
)
wjp = wjp.drop(columns=['year_str'])

print(f"Shape: {wjp.shape}")
print(f"Years: {sorted(wjp['year'].unique())}")
print(f"Countries: {wjp['country_code'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (wjp.isnull().sum() / len(wjp) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(wjp.head())

Shape: (1484, 11)
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Countries: 143

Missing values (%):
Series([], dtype: float64)
  country_name country_code  wjp_f2_absence_corruption  \
0      Albania          ALB                   0.311833   
1    Argentina          ARG                   0.474730   
2    Australia          AUS                   0.896729   
3      Austria          AUT                   0.773411   
4   Bangladesh          BGD                   0.290016   

   wjp_f3_open_government  wjp_f4_fundamental_rights  wjp_f5_order_security  \
0                0.444956                   0.631233               0.729112   
1                0.479807                   0.630741               0.595719   
2                0.840763                   0.842861               0.858811   
3                0.802236                   0.824339  

In [11]:
# Save to processed
output_path = os.path.join(PROCESSED_DIR, "wjp_clean.csv")
wjp.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {wjp.shape}")

# Derive metadata from data — no hardcoding
latest_year = str(int(wjp['year'].max()))
data_as_of_date = f"{WJP_YEAR}"

# Update download log
update_entry(
    "WJP",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="wjp_clean.csv",
    latest_available_version=latest_year,
    notes=f"Historical Data sheet. Factors 2-8 plus sub-indicator 6.5. Automated download from worldjusticeproject.org. 2017 edition not published."
)

print_entry("WJP")

Written: C:\Users\mjbou\governance-framework\data\processed\wjp_clean.csv
Shape: (1484, 11)
[download_log] Updated entry for WJP
  source_id: WJP
  last_attempted_date: 2026-05-21
  last_successful_download_date: 2026-05-21
  data_as_of_date: 2025
  local_filename: wjp_clean.csv
  latest_available_version: 2025
  no_update_reason: nan
  notes: Historical Data sheet. Factors 2-8 plus sub-indicator 6.5. Automated download from worldjusticeproject.org. 2017 edition not published.
